# SmokeGuard AI - Train YOLOv8 on D-Fire

1. Download D-Fire from https://github.com/gaiasd/DFireDataset (Google Drive link in the README) and upload the folder to your Google Drive as `MyDrive/D-Fire` (must contain `train/` and `test/`, each with `images/` and `labels/`).
2. Runtime > Change runtime type > **GPU (T4)**.
3. Run all cells. At the end you get `best.pt` and `metrics.json` -> put both in `backend/models/`.

Classes in D-Fire: `0 = smoke`, `1 = fire`.

In [ ]:
!pip -q install ultralytics
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/D-Fire'
import os; print(os.listdir(DATA))

In [ ]:
yaml = f'''path: {DATA}
train: train/images
val: test/images
names:
  0: smoke
  1: fire
'''
open('/content/dfire.yaml','w').write(yaml)
print(yaml)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')   # use yolov8n.pt for faster/smaller
model.train(data='/content/dfire.yaml', epochs=50, imgsz=640, batch=16,
            project='/content/runs', name='smokeguard', patience=15)

In [ ]:
best = YOLO('/content/runs/smokeguard/weights/best.pt')
m = best.val(data='/content/dfire.yaml')
import json
metrics = {'map50': float(m.box.map50), 'map50_95': float(m.box.map),
           'precision': float(m.box.mp), 'recall': float(m.box.mr),
           'classes': ['smoke','fire'], 'dataset': 'D-Fire', 'model': 'yolov8s', 'epochs': 50}
json.dump(metrics, open('/content/metrics.json','w'), indent=2)
print(metrics)

In [ ]:
# Confusion matrix + curves are saved here - put them in your report:
from IPython.display import Image, display
for f in ['confusion_matrix.png','PR_curve.png','results.png']:
    display(Image('/content/runs/smokeguard/'+f))

In [ ]:
import shutil
shutil.copy('/content/runs/smokeguard/weights/best.pt','/content/drive/MyDrive/best.pt')
shutil.copy('/content/metrics.json','/content/drive/MyDrive/metrics.json')
from google.colab import files
files.download('/content/runs/smokeguard/weights/best.pt'); files.download('/content/metrics.json')